# 03 — ML Model Training + Evaluation

Trains classification and regression models using `src/models/` modules.

**Models:** XGBoost, LightGBM, Random Forest (all via sklearn Pipeline)
**Tuning:** Per-class threshold optimization for improved F1
**Evaluation:** Accuracy, Macro F1, Cohen's Kappa, Fairness by Borough, SHAP

All training logic lives in `run_training.py` and `src/models/`. This notebook demonstrates the results.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from src.config import MODELS_DIR, PRICE_ZONE_LABELS, RANDOM_SEED, TEST_SIZE
from src.config import NUMERIC_FEATURES, ONEHOT_FEATURES, TARGET_ENCODED_FEATURES
from src.models.evaluate import evaluate_classifier, evaluate_regressor, evaluate_fairness_by_group
from src.models.threshold import optimize_thresholds, predict_with_thresholds
from src.utils.validation import assert_no_leakage
from run_training import prepare_data, get_feature_df

## Prepare Data + Train/Test Split

In [ ]:
df, y_zone, y_log_price, borough = prepare_data()
X = get_feature_df(df)

le = LabelEncoder()
y_zone_encoded = le.fit_transform(y_zone)

X_train, X_test, yz_train, yz_test, yp_train, yp_test, _, borough_test = train_test_split(
    X, y_zone_encoded, y_log_price, borough,
    test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_zone_encoded,
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Features: {list(X_train.columns)}")

## Load Trained Models + Evaluate

In [ ]:
# Classification
clf = joblib.load(MODELS_DIR / "price_zone_best.joblib")
y_pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)

clf_metrics = evaluate_classifier(yz_test, y_pred, PRICE_ZONE_LABELS)
print("=== Classification (argmax) ===")
print(f"Accuracy:    {clf_metrics['accuracy']:.4f}")
print(f"Macro F1:    {clf_metrics['macro_f1']:.4f}")
print(f"Cohen Kappa: {clf_metrics['cohen_kappa']:.4f}")
print(f"\n{classification_report(yz_test, y_pred, target_names=PRICE_ZONE_LABELS)}")

## Threshold Tuning (per-class optimization)

In [ ]:
thresholds, tuned_f1 = optimize_thresholds(proba, yz_test, PRICE_ZONE_LABELS)
y_tuned = predict_with_thresholds(proba, thresholds, PRICE_ZONE_LABELS)
tuned_metrics = evaluate_classifier(yz_test, y_tuned, PRICE_ZONE_LABELS)

print("=== Classification (threshold-tuned) ===")
print(f"Thresholds: {thresholds}")
print(f"Macro F1:   {clf_metrics['macro_f1']:.4f} -> {tuned_metrics['macro_f1']:.4f} (+{tuned_metrics['macro_f1'] - clf_metrics['macro_f1']:.4f})")
print(f"\n{classification_report(yz_test, y_tuned, target_names=PRICE_ZONE_LABELS)}")

## Regression Results

In [ ]:
reg = joblib.load(MODELS_DIR / "price_regressor_best.joblib")
y_price_pred = reg.predict(X_test)
reg_metrics = evaluate_regressor(yp_test.values, y_price_pred, log_target=True)

print("=== Regression (LOG_PRICE target, no leakage) ===")
print(f"R2:       {reg_metrics['r2']:.4f}")
print(f"RMSE:     {reg_metrics['rmse']:.4f}")
print(f"MAE USD:  ${reg_metrics.get('mae_usd', 0):,.0f}")
print(f"MAPE:     {reg_metrics.get('mape', 0)*100:.1f}%")

## Fairness Analysis by Borough

In [ ]:
fairness = evaluate_fairness_by_group(yz_test, y_pred, borough_test)
print("=== Fairness: Macro F1 by Borough ===")
for borough_name, score in sorted(fairness.items(), key=lambda x: -x[1]):
    bar = "#" * int(score * 40)
    print(f"  {borough_name:20s} {score:.3f}  {bar}")